## Part 1: Exploratory Data Analysis (EDA)

### Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
sns.set(style='ticks')

df = pd.read_csv("bioactivity_preprocessed_data.csv")
print("Original shape:", df.shape)

### Data Cleaning

In [ ]:
df = df.dropna(subset=[
    "molecule_chembl_id",
    "canonical_smiles",
    "standard_value"
])
df["standard_value"] = pd.to_numeric(df["standard_value"], errors="coerce")
df = df.dropna(subset=["standard_value"])
df = df[df["bioactivity_class"] != "intermediate"]
print("After cleaning:", df.shape)

### Deduplication and pIC50 Calculation

In [ ]:
df_clean = (
    df
    .groupby("canonical_smiles", as_index=False)
    .agg({
        "molecule_chembl_id": "first",
        "standard_value": "median",
        "bioactivity_class": "first"
    })
)
print("Final dataset size:", df_clean.shape)

df_clean["standard_value"] = df_clean["standard_value"].clip(lower=1e-9)
df_clean["pIC50"] = -np.log10(df_clean["standard_value"] * 1e-9)
df_clean = df_clean.replace([np.inf, -np.inf], np.nan).dropna(subset=["pIC50"])

threshold = 6
df_clean["bioactivity_class"] = np.where(
    df_clean["pIC50"] >= threshold, "active", "inactive"
)

df_clean["pIC50"].describe()

### pIC50 Distribution

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(df_clean["pIC50"], bins=30, color='skyblue', edgecolor='black')
plt.xlabel("pIC50")
plt.ylabel("Count")
plt.title("pIC50 Distribution")
plt.savefig("histogram_pIC50.pdf")
plt.show()

### Active vs Inactive Class Counts

In [ ]:
plt.figure(figsize=(5.5, 5.5))
sns.countplot(x="bioactivity_class", data=df_clean, edgecolor="black")
plt.xlabel("Bioactivity Class")
plt.ylabel("Frequency")
plt.title("Active vs Inactive Counts")
plt.savefig("barplot_bioactivity_class.pdf")
plt.show()

### Save Cleaned Data for Next Steps

In [ ]:
df_clean.to_csv("df_clean.csv", index=False)
print("Saved df_clean.csv")

## Part 2: Lipinski Descriptors

### Setup

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
sns.set(style='ticks')

df_clean = pd.read_csv("df_clean.csv")
print("Loaded:", df_clean.shape)

### Install and Import RDKit

!pip install rdkit

In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors

### Compute Lipinski Descriptors

In [ ]:
def lipinski(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return pd.Series([
        Descriptors.MolWt(mol),
        Descriptors.MolLogP(mol),
        Descriptors.NumHDonors(mol),
        Descriptors.NumHAcceptors(mol)
    ])

lipinski_desc = df_clean["canonical_smiles"].apply(lipinski)
lipinski_desc.columns = ["MW", "LogP", "NumHDonors", "NumHAcceptors"]

df_combined = pd.concat([df_clean, lipinski_desc], axis=1)
df_combined.to_csv("df_lipinski.csv", index=False)
print(df_combined.head())

### Boxplots: Lipinski Descriptors vs Bioactivity Class

In [ ]:
for col in ["MW", "LogP", "NumHDonors", "NumHAcceptors"]:
    plt.figure(figsize=(5.5, 5.5))
    sns.boxplot(x="bioactivity_class", y=col, data=df_combined, hue="bioactivity_class")
    plt.xlabel("Bioactivity Class")
    plt.ylabel(col)
    plt.title(f"{col} vs Bioactivity Class")
    plt.savefig(f"boxplot_{col}.pdf")
    plt.show()

### Scatter Plot: MW vs LogP

In [ ]:
plt.figure(figsize=(6, 6))
sns.scatterplot(
    x="MW", y="LogP", data=df_combined,
    hue="bioactivity_class", size="pIC50",
    alpha=0.7, edgecolor="black"
)
plt.xlabel("MW")
plt.ylabel("LogP")
plt.title("MW vs LogP by Bioactivity")
plt.savefig("scatter_MW_vs_LogP.pdf")
plt.show()

### Mann–Whitney U Test

In [ ]:
def mannwhitney(feature):
    active   = df_combined[df_combined.bioactivity_class == "active"][feature]
    inactive = df_combined[df_combined.bioactivity_class == "inactive"][feature]
    stat, p  = mannwhitneyu(active, inactive)
    print(f"{feature}")
    print("Statistic:", stat)
    print("p-value:", p)
    print("------------------")

features = ["pIC50", "MW", "LogP", "NumHDonors", "NumHAcceptors"]
for f in features:
    mannwhitney(f)

## Part 3: Molecular Fingerprints

### Setup

In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import MACCSkeys

df_clean = pd.read_csv("df_clean.csv")
print("Loaded:", df_clean.shape)

### Install PaDEL-Descriptor

!pip install padelpy

### Generate PaDEL Descriptors

In [ ]:
df_clean[["canonical_smiles", "molecule_chembl_id"]].to_csv(
    "smiles_chembl.smi",
    sep="\t",
    index=False,
    header=False
)

from padelpy import padeldescriptor

padeldescriptor(
    mol_dir="smiles_chembl.smi",
    d_file="padel_descriptors.csv",
    retainorder=True
)

padel = pd.read_csv("padel_descriptors.csv")
print("PaDEL descriptor shape:", padel.shape)

### Generate MACCS Fingerprints

In [ ]:
fps = []
for smi in df_clean["canonical_smiles"]:
    mol = Chem.MolFromSmiles(smi)
    fp  = MACCSkeys.GenMACCSKeys(mol)
    fps.append(list(fp.ToBitString()))

df_maccs = pd.DataFrame(fps)
df_maccs.to_csv("MACCS_fingerprints.csv", index=False)
print("MACCS fingerprint shape:", df_maccs.shape)